# Logistic Regression: Premier League Match Outcomes
Predict home wins (`H`), draws (`D`), and away wins (`A`) using multinomial logistic regression. Use the shared 18 pre-match features and the same chronological train/test split as `random_forest.ipynb` and `gradient_boosting.ipynb`.

A pipeline fits `StandardScaler` and the classifier together. During chronological cross validation, scaling is learned from each fold's training matches only. After selecting settings, the pipeline is refitted on all training matches; the test season is excluded from parameter selection.

Betting odds are retained only for the benchmark and never become classifier inputs. Team form updates with prior results, including earlier matches in the test season, while the fitted model remains fixed throughout that season.

Documentation: [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html), [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).


In [1]:
import pandas as pd
import numpy as np
from match_features import MatchFeatureBuilder


## Load the Data

In [2]:
# Combine all six seasons into one DataFrame.
csv_files = [
    "20-21.csv",
    "21-22.csv",
    "22-23.csv",
    "23-24.csv",
    "24-25.csv",
    "25-26.csv",
]

# Use the same feature configuration in each model notebook.
feature_builder = MatchFeatureBuilder(windows=(5, 20), min_matches=5)
df = feature_builder.load_csvs(csv_files, extra_columns=["AvgH", "AvgD", "AvgA"])

print(f"Loaded {len(df):,} matches and {df.shape[1]} columns.")
df.head()


Loaded 2,280 matches and 27 columns.


,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,AF,HC,AC,HY,AY,HR,AR,AvgH,AvgD,AvgA
0,E0,12/09/2020,12:30,Fulham,Arsenal,0,3,A,0,1,...,12,2,3,2,2,0,0,5.94,4.34,1.55
1,E0,12/09/2020,15:00,Crystal Palace,Southampton,1,0,H,1,0,...,11,7,3,2,1,0,0,3.18,3.22,2.39
2,E0,12/09/2020,17:30,Liverpool,Leeds,4,3,H,3,2,...,6,9,0,1,0,0,0,1.30,5.96,9.68
3,E0,12/09/2020,20:00,West Ham,Newcastle,0,2,A,0,0,...,7,8,7,2,2,0,0,2.15,3.48,3.42
4,E0,13/09/2020,14:00,West Brom,Leicester,0,3,A,0,0,...,9,2,5,1,1,0,0,3.87,3.57,1.97


## Add the Shared Features
Use the five-match and up-to-20-match averages plus home-minus-away differences. Build histories before splitting by date and remove matches with insufficient prior history.

In [3]:
# Build every team's history before splitting into train and test sets.
rows_before = len(df)
df = feature_builder.add_features(df)
feature_columns = feature_builder.feature_columns.copy()

print(f"Added {len(feature_columns)} features.")
print(f"Dropped {rows_before - len(df):,} matches with insufficient history.")
print(f"Remaining: {len(df):,} matches and {df.shape[1]} columns.")
df[["Date", "HomeTeam", "AwayTeam", "FTR"] + feature_columns].head()


Added 18 features.
Dropped 91 matches with insufficient history.
Remaining: 2,189 matches and 45 columns.


,Date,HomeTeam,AwayTeam,FTR,home_points_last5,home_goals_scored_last5,home_goals_conceded_last5,home_points_last20,home_goals_scored_last20,home_goals_conceded_last20,...,away_goals_conceded_last5,away_points_last20,away_goals_scored_last20,away_goals_conceded_last20,points_difference_last5,goals_scored_difference_last5,goals_conceded_difference_last5,points_difference_last20,goals_scored_difference_last20,goals_conceded_difference_last20
0,2020-10-24,Fulham,Crystal Palace,A,0.2,0.8,2.4,0.2,0.8,2.4,...,1.6,1.4,1.2,1.6,-1.2,-0.4,0.8,-1.2,-0.4,0.8
1,2020-10-24,Liverpool,Sheffield United,H,2.0,2.6,2.6,2.0,2.6,2.6,...,1.4,0.2,0.4,1.4,1.8,2.2,1.2,1.8,2.2,1.2
2,2020-10-25,Southampton,Everton,H,1.4,1.6,1.8,1.4,1.6,1.8,...,1.4,2.6,2.8,1.4,-1.2,-1.2,0.4,-1.2,-1.2,0.4
3,2020-10-25,Wolves,Newcastle,D,1.8,1.0,1.4,1.8,1.0,1.4,...,1.8,1.4,1.4,1.8,0.4,-0.4,-0.4,0.4,-0.4,-0.4
4,2020-10-25,Arsenal,Leicester,A,1.8,1.6,1.2,1.8,1.6,1.2,...,1.6,1.8,2.4,1.6,0.0,-0.8,-0.4,0.0,-0.8,-0.4


## Prepare the Data for Machine Learning
Use the 12 averages and six home-minus-away differences from the five-match and 20-match windows as inputs and `FTR` as the label (`H` = home win, `D` = draw, `A` = away win).

Train on seasons 2020-21 through 2024-25 and reserve 2025-26 for testing. Keep matches in time order. For each test match, the averages use results available before kickoff, including earlier matches in the test season. This evaluates predicting matches one at a time, with the fitted model held fixed throughout the test season.


In [4]:
# July 1 separates the 2024-25 and 2025-26 seasons.
test_start = pd.Timestamp("2025-07-01")
train_set = df.loc[df["Date"] < test_start].copy()
test_set = df.loc[df["Date"] >= test_start].copy()

X_train = train_set[feature_columns].copy()
y_train = train_set["FTR"].copy()
X_test = test_set[feature_columns].copy()
y_test = test_set["FTR"].copy()

print("Training features:", X_train.shape)
print("Test features:", X_test.shape)
X_train.head()


Training features: (1814, 18)
Test features: (375, 18)


,home_points_last5,home_goals_scored_last5,home_goals_conceded_last5,home_points_last20,home_goals_scored_last20,home_goals_conceded_last20,away_points_last5,away_goals_scored_last5,away_goals_conceded_last5,away_points_last20,away_goals_scored_last20,away_goals_conceded_last20,points_difference_last5,goals_scored_difference_last5,goals_conceded_difference_last5,points_difference_last20,goals_scored_difference_last20,goals_conceded_difference_last20
0,0.2,0.8,2.4,0.2,0.8,2.4,1.4,1.2,1.6,1.4,1.2,1.6,-1.2,-0.4,0.8,-1.2,-0.4,0.8
1,2.0,2.6,2.6,2.0,2.6,2.6,0.2,0.4,1.4,0.2,0.4,1.4,1.8,2.2,1.2,1.8,2.2,1.2
2,1.4,1.6,1.8,1.4,1.6,1.8,2.6,2.8,1.4,2.6,2.8,1.4,-1.2,-1.2,0.4,-1.2,-1.2,0.4
3,1.8,1.0,1.4,1.8,1.0,1.4,1.4,1.4,1.8,1.4,1.4,1.8,0.4,-0.4,-0.4,0.4,-0.4,-0.4
4,1.8,1.6,1.2,1.8,1.6,1.2,1.8,2.4,1.6,1.8,2.4,1.6,0.0,-0.8,-0.4,0.0,-0.8,-0.4


## Tune a Logistic Regression Classifier
Use L2 regularization with the `lbfgs` solver. Search ten values of `C` (inverse regularization strength) and two class-weight settings: no weighting and `balanced`. Smaller `C` means stronger regularization; balanced weights give less common outcomes more weight during fitting.

This small grid contains 20 configurations evaluated across the same three expanding chronological folds (60 validation fits plus a refit). Select the configuration using accuracy. Keep both scaling and classification inside the pipeline to prevent validation leakage.


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

log_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        solver="lbfgs", max_iter=2000, random_state=42
    )),
])

param_grid = {
    "classifier__C": [0.0001, 0.001, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 100],
    "classifier__class_weight": [None, "balanced"],
}

time_split = TimeSeriesSplit(n_splits=3)
grid_search = GridSearchCV(
    log_pipeline,
    param_grid=param_grid,
    cv=time_split,
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True,
    error_score="raise",
)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print(f"Best validation accuracy: {grid_search.best_score_:.2%}")

# The complete selected pipeline is already refitted on all training matches.
log_reg = grid_search.best_estimator_


Best parameters: {'classifier__C': 0.01, 'classifier__class_weight': None}
Best validation accuracy: 55.56%


## Inspect Search Results
Reuse the fold scores from the search and inspect the leading configurations. The best validation score is a selection score and can be optimistic after comparing multiple configurations.

In [6]:
log_scores = np.array([
    grid_search.cv_results_[f"split{fold}_test_score"][grid_search.best_index_]
    for fold in range(time_split.n_splits)
])

print("Scores:", log_scores)
print("Mean:", log_scores.mean())
print("Standard deviation:", log_scores.std())

search_results = pd.DataFrame(grid_search.cv_results_)
search_results.sort_values("rank_test_score")[[
    "mean_test_score", "std_test_score", "mean_train_score", "params",
]].head(10)


Scores: [0.54304636 0.57836645 0.54525386]
Mean: 0.5555555555555555
Standard deviation: 0.01615489212663732


,mean_test_score,std_test_score,mean_train_score,params
4,0.555556,0.016155,0.532303,"{'classifier__C': 0.01, 'classifier__class_wei..."
6,0.551141,0.013039,0.536703,"{'classifier__C': 0.03, 'classifier__class_wei..."
2,0.550405,0.015748,0.520938,"{'classifier__C': 0.001, 'classifier__class_we..."
8,0.541575,0.018411,0.536582,"{'classifier__C': 0.1, 'classifier__class_weig..."
10,0.540839,0.019075,0.536584,"{'classifier__C': 0.3, 'classifier__class_weig..."
14,0.539367,0.021045,0.534996,"{'classifier__C': 3, 'classifier__class_weight..."
12,0.539367,0.021045,0.535484,"{'classifier__C': 1, 'classifier__class_weight..."
16,0.539367,0.021045,0.534996,"{'classifier__C': 10, 'classifier__class_weigh..."
18,0.539367,0.021045,0.534996,"{'classifier__C': 100, 'classifier__class_weig..."
1,0.531273,0.013528,0.498797,"{'classifier__C': 0.0001, 'classifier__class_w..."


## Evaluate the Selected Model on the Test Set
Apply the complete pipeline directly to the original test features. Its scaler uses only the training-data statistics. Test results are reported after model selection.

In [7]:
from sklearn.metrics import accuracy_score, classification_report

y_train_pred = log_reg.predict(X_train)
y_predict_test = log_reg.predict(X_test)
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_predict_test)

print(f"Training accuracy: {train_accuracy:.2%}")
print(f"Test accuracy: {test_accuracy:.2%}")
print()
print(classification_report(
    y_test, y_predict_test,
    labels=["H", "D", "A"],
    target_names=["Home win", "Draw", "Away win"],
    zero_division=0,
))


Training accuracy: 55.07%
Test accuracy: 49.07%

              precision    recall  f1-score   support

    Home win       0.52      0.76      0.62       159
        Draw       0.00      0.00      0.00       102
    Away win       0.44      0.55      0.49       114

    accuracy                           0.49       375
   macro avg       0.32      0.44      0.37       375
weighted avg       0.36      0.49      0.41       375



In [8]:
from sklearn.metrics import confusion_matrix

# Rows are actual results; columns are predicted results.
conf_mx = confusion_matrix(y_test, y_predict_test, labels=["H", "D", "A"])
pd.DataFrame(
    conf_mx,
    index=pd.Index(["Home win", "Draw", "Away win"], name="Actual"),
    columns=pd.Index(["Home win", "Draw", "Away win"], name="Predicted"),
)


Predicted,Home win,Draw,Away win
Actual,,,
Home win,121,0,38
Draw,60,0,42
Away win,51,0,63


## Model Coefficients
Show the coefficients for the standardized inputs for each outcome. They contribute to the class scores that are converted into probabilities. Positive coefficients increase that class's score as the feature increases, holding other inputs fixed.

The averages and their differences are correlated, so individual coefficients should not be interpreted as unique or causal effects.


In [9]:
classifier = log_reg.named_steps["classifier"]
coefficients = pd.DataFrame(
    classifier.coef_.T,
    index=pd.Index(feature_columns, name="Feature"),
    columns=classifier.classes_,
).rename(columns={"H": "Home win", "D": "Draw", "A": "Away win"})
coefficients


,Away win,Draw,Home win
Feature,,,
home_points_last5,-0.040295,0.017769,0.022526
home_goals_scored_last5,-0.010184,0.008458,0.001726
home_goals_conceded_last5,0.003389,0.013588,-0.016977
home_points_last20,-0.119821,0.006639,0.113182
home_goals_scored_last20,-0.047546,-0.036038,0.083584
home_goals_conceded_last20,-0.009083,0.011365,-0.002282
away_points_last5,-0.075006,0.031001,0.044005
away_goals_scored_last5,0.014950,-0.013256,-0.001694
away_goals_conceded_last5,-0.060231,-0.002459,0.062690


## Compare with the Betting Favorite and Home-Win Baseline
Choose the lowest market-average pre-closing odds among `AvgH`, `AvgD`, and `AvgA`. Predictions assume these odds snapshots are available. For an exact tie use the first outcome in the fixed order `H`, `D`, `A`.

Exclude matches with any missing, nonfinite, or invalid odds (less than or equal to 1) from all three methods in this table. All comparison accuracies therefore use the same matches. Source: [Football-Data column definitions](https://football-data.co.uk/notes.txt).


In [10]:
odds_columns = ["AvgH", "AvgD", "AvgA"]
valid_odds = (
    test_set[odds_columns].notna().all(axis=1)
    & test_set[odds_columns].gt(1).all(axis=1)
    & np.isfinite(test_set[odds_columns]).all(axis=1)
)
comparison = test_set.loc[valid_odds].copy()
if comparison.empty:
    raise ValueError("No eligible test matches have valid odds for all three outcomes.")

outcome_odds = comparison[odds_columns].rename(columns={
    "AvgH": "H", "AvgD": "D", "AvgA": "A",
})
comparison["betting_prediction"] = outcome_odds.idxmin(axis=1)
comparison["logistic_prediction"] = log_reg.predict(comparison[feature_columns])
comparison["home_prediction"] = "H"

favorite_ties = outcome_odds.eq(outcome_odds.min(axis=1), axis=0).sum(axis=1).gt(1)
print(f"Eligible test matches: {len(test_set):,}")
print(f"Excluded for missing/invalid odds: {len(test_set) - len(comparison):,}")
print(f"Matches compared for every method: {len(comparison):,}")
print(f"Tied betting favorites: {favorite_ties.sum():,}")


Eligible test matches: 375
Excluded for missing/invalid odds: 0
Matches compared for every method: 375
Tied betting favorites: 0


In [11]:
prediction_columns = {
    "Logistic regression": "logistic_prediction",
    "Betting favorite (market average)": "betting_prediction",
    "Always home win": "home_prediction",
}
results = []
for method, column in prediction_columns.items():
    correct = int((comparison[column] == comparison["FTR"]).sum())
    accuracy = accuracy_score(comparison["FTR"], comparison[column])
    results.append({
        "Method": method,
        "Correct": correct,
        "Matches": len(comparison),
        "Accuracy (%)": round(100 * accuracy, 2),
    })

accuracy_comparison = pd.DataFrame(results).set_index("Method")
accuracy_comparison


,Correct,Matches,Accuracy (%)
Method,,,
Logistic regression,184,375,49.07
Betting favorite (market average),187,375,49.87
Always home win,159,375,42.40


## Inspect Individual Predictions

In [12]:
match_predictions = comparison[[
    "Date", "HomeTeam", "AwayTeam", "FTR",
    "logistic_prediction", "betting_prediction",
]].rename(columns={"FTR": "actual_result"})
match_predictions.head(20)


,Date,HomeTeam,AwayTeam,actual_result,logistic_prediction,betting_prediction
1814,2025-08-15,Liverpool,Bournemouth,H,H,H
1815,2025-08-16,Aston Villa,Newcastle,D,H,H
1816,2025-08-16,Brighton,Fulham,D,H,H
1817,2025-08-16,Tottenham,Burnley,H,H,H
1818,2025-08-16,Wolves,Man City,A,A,A
1819,2025-08-17,Chelsea,Crystal Palace,D,H,H
1820,2025-08-17,Nott'm Forest,Brentford,H,H,H
1821,2025-08-17,Man United,Arsenal,A,A,A
1822,2025-08-18,Leeds,Everton,H,A,H
1823,2025-08-22,West Ham,Chelsea,A,A,A
